# Day 4 — Modelling for AI

**Workshop:** Mathematical Foundations of Modern AI

Companion notebook to the Day 4 lecture notes. Three experiments, each isolating a different inductive bias:

1. **Translation equivariance.** MLP vs CNN on MNIST, both at standard pose and at shifted pose. The CNN's translation equivariance pays off when train and test no longer match exactly.
2. **Permutation equivariance.** A tiny message-passing GNN vs an MLP on a synthetic node-classification problem. The GNN uses the graph structure; the MLP cannot.
3. **Attention as content-based aggregation.** Self-attention computed from scratch on a toy sequence, with the attention weights visualized as a heatmap.

Runs on CPU, GPU optional.

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Experiment 1 — MLP vs CNN on shifted MNIST

Build both networks with comparable parameter counts. Train on standard MNIST. Then evaluate on a shifted test set, where each test image has been translated by a few pixels. The MLP has to learn translation invariance from data; the CNN gets it for free.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_ds = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root=".", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128), nn.ReLU(),
            nn.Linear(128, 64),    nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x): return self.net(x)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x): return self.head(self.conv(x))

mlp = MLP().to(device)
cnn = CNN().to(device)
print(f"MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}")
print(f"CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

In [ ]:
def train(model, n_epochs=2):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(n_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

train(mlp); train(cnn)
mlp_clean = evaluate(mlp, test_loader)
cnn_clean = evaluate(cnn, test_loader)
print(f"Standard MNIST   — MLP: {mlp_clean:.3f}, CNN: {cnn_clean:.3f}")

In [ ]:
# Build a shifted test set: each image translated by random (dx, dy) in [-4, 4]
class ShiftedMNIST(torch.utils.data.Dataset):
    def __init__(self, base, max_shift=4):
        self.base = base
        self.max_shift = max_shift
        self.rng_state = np.random.default_rng(123)
        self.shifts = self.rng_state.integers(-max_shift, max_shift+1, size=(len(base), 2))
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, y = self.base[i]
        dx, dy = self.shifts[i]
        return torch.roll(img, shifts=(int(dy), int(dx)), dims=(1, 2)), y

shifted_loader = DataLoader(ShiftedMNIST(test_ds, max_shift=4), batch_size=256)

mlp_shift = evaluate(mlp, shifted_loader)
cnn_shift = evaluate(cnn, shifted_loader)
print(f"Shifted MNIST    — MLP: {mlp_shift:.3f}, CNN: {cnn_shift:.3f}")

In [ ]:
# Visualize the comparison
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
ax.bar(x - 0.18, [mlp_clean, mlp_shift], width=0.35, label="MLP")
ax.bar(x + 0.18, [cnn_clean, cnn_shift], width=0.35, label="CNN")
ax.set_xticks(x)
ax.set_xticklabels(["Standard MNIST", "Shifted (±4 px)"])
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1)
ax.set_title("Inductive bias of translation equivariance")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

On standard MNIST the two are close. On shifted MNIST the MLP drops sharply --- it learned to recognize digits at their canonical position. The CNN drops much less, because the convolution operation itself is translation equivariant. The CNN did not have to learn that shifted-3 is still 3 from data; the architecture told it.

---

## Experiment 2 — MLP vs GNN on synthetic node classification

Build a synthetic graph with two communities. Label a few nodes from each community as training data, and ask both models to label the rest.

The MLP has only node identity as input. The GNN has node identity AND the adjacency matrix. The graph structure is the signal the MLP cannot see.

In [ ]:
# Stochastic block model: two communities, dense intra, sparse inter
n_per_comm = 20
n = 2 * n_per_comm
p_intra, p_inter = 0.5, 0.05

A = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        same = (i // n_per_comm) == (j // n_per_comm)
        p = p_intra if same else p_inter
        if rng.random() < p:
            A[i, j] = A[j, i] = 1.0

labels = np.array([0] * n_per_comm + [1] * n_per_comm)
X = np.eye(n, dtype=np.float32)         # one-hot node identity

# Pick 5 labeled nodes per community for training
rng2 = np.random.default_rng(1)
train_idx = np.concatenate([
    rng2.choice(np.where(labels == 0)[0], 5, replace=False),
    rng2.choice(np.where(labels == 1)[0], 5, replace=False),
])
test_idx = np.array([i for i in range(n) if i not in train_idx])

X_t = torch.tensor(X, device=device)
A_t = torch.tensor(A, dtype=torch.float32, device=device)
y_t = torch.tensor(labels, dtype=torch.long, device=device)

# Symmetric normalized adjacency with self-loops (GCN-style)
A_hat = A_t + torch.eye(n, device=device)
deg = A_hat.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt

print(f"Graph: {n} nodes, {int(A.sum()/2)} undirected edges")
print(f"Training set: {len(train_idx)} labeled nodes ({len(train_idx)/n:.0%} of graph)")

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, d_in, d_hidden=16, d_out=2):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
    def forward(self, X, A_norm=None):                   # A_norm ignored
        return self.fc2(F.relu(self.fc1(X)))

class SimpleGCN(nn.Module):
    """Two-layer Graph Convolutional Network (Kipf & Welling, 2017)."""
    def __init__(self, d_in, d_hidden=16, d_out=2):
        super().__init__()
        self.W1 = nn.Linear(d_in, d_hidden, bias=False)
        self.W2 = nn.Linear(d_hidden, d_out, bias=False)
    def forward(self, X, A_norm):
        H = F.relu(A_norm @ self.W1(X))   # message passing
        return A_norm @ self.W2(H)

def train_node_classifier(model, n_epochs=200):
    opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=5e-4)
    for _ in range(n_epochs):
        model.train()
        out = model(X_t, A_norm)
        loss = F.cross_entropy(out[train_idx], y_t[train_idx])
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(X_t, A_norm).argmax(dim=1)
    return float((preds[test_idx] == y_t[test_idx]).float().mean())

# Average over a few seeds for stability
mlp_accs, gcn_accs = [], []
for seed in range(5):
    torch.manual_seed(seed)
    mlp_accs.append(train_node_classifier(SimpleMLP(d_in=n).to(device)))
    torch.manual_seed(seed)
    gcn_accs.append(train_node_classifier(SimpleGCN(d_in=n).to(device)))

print(f"MLP test accuracy (mean ± std over 5 seeds): {np.mean(mlp_accs):.3f} ± {np.std(mlp_accs):.3f}")
print(f"GCN test accuracy (mean ± std over 5 seeds): {np.mean(gcn_accs):.3f} ± {np.std(gcn_accs):.3f}")

In [ ]:
# Visualize the graph with predicted vs true labels (GCN)
import networkx as nx
G = nx.from_numpy_array(A)
pos = nx.spring_layout(G, seed=0)

torch.manual_seed(0)
gcn = SimpleGCN(d_in=n).to(device)
_ = train_node_classifier(gcn)
with torch.no_grad():
    pred = gcn(X_t, A_norm).argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, color, title in zip(axes, [labels, pred], ["True community labels", "GCN predictions"]):
    nx.draw(G, pos, node_color=color, cmap="coolwarm", node_size=120, with_labels=False, ax=ax,
            edge_color="lightgray")
    nx.draw_networkx_nodes(G, pos, nodelist=list(train_idx), node_color="yellow", node_size=200,
                            edgecolors="black", ax=ax)
    ax.set_title(title)
plt.suptitle("Yellow = labeled training nodes; colored = inferred / true community"); plt.tight_layout(); plt.show()

The MLP sits around chance accuracy (50%) on the held-out nodes --- it never sees which nodes are connected to which, so the test nodes are just unseen one-hot vectors and it has no way to generalize. The GCN reaches near-perfect accuracy because each layer propagates information along the adjacency, so each unlabeled node ends up close in representation to its labeled neighbors of the same community.

The inductive bias of message passing along edges is the difference between learning and guessing.

---

## Experiment 3 — Self-attention from scratch

Take a short sequence of fake word embeddings. Compute query, key, value projections. Compute the attention weights. Visualize them.

The point is to see that attention's inductive bias is **content-based aggregation**: each output is a weighted average of inputs, where the weights depend on dot products in a learned space. No spatial structure, no order, no graph --- just similarity in the projected space.

In [ ]:
torch.manual_seed(42)
tokens = ["the", "cat", "sat", "on", "the", "mat"]
n_tok = len(tokens)
d_model = 16
d_k = 8

# Random token embeddings (in a real model these would be learned)
# Tie identical tokens to identical embeddings so 'the' looks the same in both positions
embedding = nn.Embedding(num_embeddings=4, embedding_dim=d_model)
vocab = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 2}   # 'mat' reused as 'sat' for illustration of similarity
token_ids = torch.tensor([vocab[t] if t in vocab else 0 for t in tokens])
X = embedding(token_ids)                       # (n_tok, d_model)

# Q, K, V projections (random, untrained)
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)
Q = W_Q(X)
K = W_K(X)
V = W_V(X)

# Scaled dot-product attention
scores = Q @ K.T / np.sqrt(d_k)               # (n_tok, n_tok)
A = F.softmax(scores, dim=-1)                  # row-normalized
Y = A @ V                                      # (n_tok, d_k)

print("Tokens:", tokens)
print(f"Q shape: {Q.shape}, K shape: {K.shape}, V shape: {V.shape}")
print(f"Attention matrix A shape: {A.shape}")
print(f"Output Y shape: {Y.shape}")

In [ ]:
# Visualize the attention matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A.detach().numpy(), cmap="Blues", vmin=0)
ax.set_xticks(range(n_tok)); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(n_tok)); ax.set_yticklabels(tokens)
ax.set_xlabel("Attended to (key)")
ax.set_ylabel("Querying token")
ax.set_title("Self-attention weights (untrained random projection)")
for i in range(n_tok):
    for j in range(n_tok):
        ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center",
                color="white" if A[i, j] > 0.2 else "black", fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

Even with untrained random Q/K/V projections, the structure shows: 

- The two occurrences of `the` attend identically because they have identical embeddings.
- Tokens that share an underlying embedding ID (we deliberately tied `mat` to the same embedding as `sat`) end up attending to each other.

In a trained Transformer, $W_Q, W_K, W_V$ are learned so that the attention patterns become useful: nouns attend to their adjectives, pronouns attend to their referents, etc. The mathematical operation is the same five-line computation you see above; everything else is scale and training data.

---

## What you have shown

Three architectural priors, three direct measurements:

- **Translation equivariance (CNNs)** keeps performance up under shifts that destroy an MLP.
- **Permutation equivariance over graphs (GCNs)** turns a hopeless 50% MLP into a near-perfect classifier.
- **Content-based aggregation (attention)** computes weighted averages whose weights are determined by similarity in a learned space.

None of these came from clever loss design, optimization tricks, or more data. They came from choosing an architecture whose mathematical structure matches the structure of the problem. That is what inductive bias buys you, and it is why architecture choice still matters even in an era of large-scale general-purpose models.

Day 5 closes the workshop with the only remaining question: how do we know the model is actually learning the right thing?